# **Automatic Alternate Abbreviation Annotation:** Analyzing results of LLM predictions

This notebook analyzes LLM predictions from the Alternate Abbreviation annotation workflow by loading previously stored LLM runs and computing performance, coverage, and system-level metrics. It also generates evaluation datasets to support troubleshooting and review of model predictions. For more information on Alternate Abbreviations and the overall workflow, see [README](alternate_abbreviation_README).

In [1]:
import math
from pathlib import Path
import pickle
import polars as pl

from alt_abbrev_models import RunResult

In [2]:
def create_analysis_summary(cm: pl.DataFrame) -> pl.DataFrame:
    """Compute per-class recall-style summary from a confusion matrix.

    :param cm: Confusion matrix (square DataFrame)
    :return: Summary DataFrame per class
    """
    rows = []

    classes = cm.columns[1:]

    for cls in classes:
        row = cm.filter(pl.col("gt") == cls)

        if row.height == 0:
            numerator = 0
            denominator = 0
        else:
            numerator = row.select(pl.col(cls)).item()

            denominator = row.select(pl.sum_horizontal(classes)).item()

        rows.append(
            {
                "consensus_w_curator": cls,
                "numerator": numerator,
                "denominator": denominator,
                "percentage": (
                    (numerator or 0) / denominator * 100
                    if denominator is not None and denominator > 0
                    else 0.0
                ),
            }
        )

    return pl.DataFrame(rows)

In [3]:
def compute_overall_accuracy(cm: pl.DataFrame) -> float:
    """Compute overall accuracy from a confusion matrix.

    :param cm: Confusion matrix DataFrame (square matrix)
    :return: Accuracy as a float between 0 and 1
    """
    classes = cm.columns[1:]

    total = cm.select(pl.sum_horizontal(classes)).to_series().sum()

    correct = 0

    for cls in classes:
        row = cm.filter(pl.col("gt") == cls)

        if row.height > 0:
            correct += row.select(pl.col(cls)).item() or 0

    return correct / total if total > 0 else math.nan

In [4]:
def boolean_confusion_matrix(
    df: pl.DataFrame, gt_col: str, pred_col: str
) -> pl.DataFrame:
    """Compute a boolean confusion matrix in the traditional matrix style.

    Parameters:
    - df: pandas DataFrame
    - gt_col: str, name of the ground truth column
    - pred_col: str, name of the predicted column

    Returns:
    - confusion_matrix: pandas DataFrame with row/column labels
    """

    df_clean = df.filter(
        pl.col(gt_col).is_not_null() & pl.col(pred_col).is_not_null()
    ).with_columns(
        [
            pl.col(gt_col).cast(pl.Boolean).alias("gt"),
            pl.col(pred_col).cast(pl.Boolean).alias("pred"),
        ]
    )

    return (
        df_clean.group_by(["gt", "pred"])
        .len()
        .pivot(
            values="len",
            index="gt",
            on="pred",
            aggregate_function="sum",
        )
        .fill_null(0)
    )

In [5]:
def create_system_level_predictions(
    df: pl.DataFrame,
    pred_col: str,
    system_pred_col: str = "llm_system",
) -> pl.DataFrame:
    """Create a system-level prediction column where null predictions
    are treated as False.

    :param df: Input dataframe
    :param pred_col: Original prediction column
    :param system_pred_col: Name of new system-level prediction column
    :return: DataFrame with added system-level prediction column
    """
    return df.with_columns(pl.col(pred_col).fill_null(False).alias(system_pred_col))

In [6]:
def compute_coverage(
    df: pl.DataFrame,
    pred_col: str,
) -> float:
    """Compute proportion of rows with non-null predictions. Low coverage is not inherently bad, as it just shows the ration of samples that were evaluated by rules based methods compared to by LLM.

    :param df: Input dataframe
    :param pred_col: Original prediction column
    :return: Proportion of rows with llm predictions
    """
    return df.select(pl.col(pred_col).is_not_null().mean()).item()

In [7]:
def analyze_results(
    df: pl.DataFrame,
) -> tuple[
    pl.DataFrame,
    float,
    pl.DataFrame,
    pl.DataFrame,
    float,
    pl.DataFrame,
    float,
]:
    """Evaluate LLM predictions against ground truth using:
    1. Conditional LLM-only evaluation
    2. System-level evaluation (null -> False)

    :param df: Input dataframe
    :return: Tuple containing:
    - LLM match analysis summary DataFrame
    - LLM overall accuracy (float)
    - LLM precision/recall/f1 metrics DataFrame
    - LLM coverage (float)
    - System-level match analysis summary DataFrame
    - System-level overall accuracy (float)
    - System-level precision/recall/f1 metrics DataFrame
    """

    analysis_df = df.clone()

    # CONDITIONAL LLM EVALUATION

    llm_cm = boolean_confusion_matrix(
        analysis_df,
        "gt",
        "llm",
    )

    TN = llm_cm.filter(~pl.col("gt"))["false"].item()
    FP = llm_cm.filter(~pl.col("gt"))["true"].item()
    FN = llm_cm.filter(pl.col("gt"))["false"].item()
    TP = llm_cm.filter(pl.col("gt"))["true"].item()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    llm_metrics_df = pl.DataFrame(
        {
            "evaluation_type": ["conditional_llm"],
            "precision": [precision],
            "recall": [recall],
            "f1": [f1],
            "TP": [TP],
            "FP": [FP],
            "TN": [TN],
            "FN": [FN],
        }
    )

    llm_match_analysis_summary = create_analysis_summary(llm_cm)

    llm_accuracy = compute_overall_accuracy(llm_cm)

    llm_coverage = compute_coverage(
        analysis_df,
        "llm",
    )

    # SYSTEM-LEVEL EVALUATION (null -> False)

    system_df = create_system_level_predictions(
        analysis_df,
        pred_col="llm",
        system_pred_col="llm_system",
    )

    system_cm = boolean_confusion_matrix(
        system_df,
        "gt",
        "llm_system",
    )

    system_TN = system_cm.filter(~pl.col("gt"))["false"].item()
    system_FP = system_cm.filter(~pl.col("gt"))["true"].item()
    system_FN = system_cm.filter(pl.col("gt"))["false"].item()
    system_TP = system_cm.filter(pl.col("gt"))["true"].item()

    system_precision = (
        system_TP / (system_TP + system_FP) if (system_TP + system_FP) > 0 else 0.0
    )

    system_recall = (
        system_TP / (system_TP + system_FN) if (system_TP + system_FN) > 0 else 0.0
    )

    system_f1 = (
        2 * system_precision * system_recall / (system_precision + system_recall)
        if (system_precision + system_recall) > 0
        else 0.0
    )

    system_metrics_df = pl.DataFrame(
        {
            "evaluation_type": ["system_level"],
            "precision": [system_precision],
            "recall": [system_recall],
            "f1": [system_f1],
            "TP": [system_TP],
            "FP": [system_FP],
            "TN": [system_TN],
            "FN": [system_FN],
        }
    )

    system_match_analysis_summary = create_analysis_summary(system_cm)

    system_accuracy = compute_overall_accuracy(system_cm)

    return (
        llm_match_analysis_summary,
        llm_accuracy,
        llm_metrics_df,
        llm_coverage,
        system_match_analysis_summary,
        system_accuracy,
        system_metrics_df,
    )

In [8]:
def build_evaluation_df(
    df: pl.DataFrame, stored_runs: list, run_idx: int = -1
) -> pl.DataFrame:
    """Attach LLM outputs + diagnostics back onto original dataframe.

    :param df: Original input dataframe
    :param stored_runs: List of stored runs containing LLM outputs
    :param run_idx: Index of the run to use from stored_runs (default: -1 for last run)
    :return: DataFrame with LLM outputs and diagnostics attached"""

    run = stored_runs[run_idx]

    results = run["results"]

    eval_rows = []

    for row, res in zip(df.iter_rows(named=True), results):
        llm_label = getattr(res, "llm_annotation", None)

        matched_rule = getattr(res, "matched_rule", None)

        eval_rows.append(
            {
                **row,
                "llm": llm_label,
                "matched_rule": matched_rule,
            }
        )

    return pl.DataFrame(eval_rows)

In [9]:
def add_correctness_column(df: pl.DataFrame) -> pl.DataFrame:
    """Add a boolean column indicating whether the LLM prediction matches the ground truth.

    :param df: Input DataFrame with 'gt' and 'llm' columns
    :return: DataFrame with added 'llm_correct' column
    """
    return df.with_columns(
        (pl.col("llm") == pl.col("alternate_abbreviation_status")).alias("llm_correct")
    )

In [10]:
def build_eval_df(
    df: pl.DataFrame,
    stored_runs: list,
    temperature: float,
    run_idx: int,
) -> pl.DataFrame:
    """Build evaluation DataFrame by attaching LLM outputs and diagnostics to the original dataframe for a specific run.

    :param df: Original input dataframe
    :param stored_runs: List of stored runs containing LLM outputs
    :param temperature: Temperature of the run to select
    :param run_idx: Index of the run to use from stored_runs (default: -1 for last run)
    :return: DataFrame with LLM outputs and diagnostics attached, plus correctness column
    """

    run = next(
        (
            r
            for r in stored_runs
            if r["temperature"] == temperature and r["run_idx"] == run_idx
        ),
        None,
    )

    if run is None:
        available = [(r["temperature"], r["run_idx"]) for r in stored_runs]
        raise ValueError(
            f"No run found for (temperature={temperature}, run_idx={run_idx}). "
            f"Available runs: {available}"
        )

    results = run["results"]

    eval_rows = []

    for row, res in zip(df.iter_rows(named=True), results):
        eval_rows.append(
            {
                **row,
                "llm": res.get("llm_annotation"),
                "matched_rule": res.get("matched_rule"),
                "temperature": temperature,
                "run_idx": run_idx,
            }
        )

    eval_df = pl.DataFrame(eval_rows)

    return add_correctness_column(eval_df)

## Load stored runs from alt_abbrev_llm_1_run_samples notebook

In [11]:
ALT_ABBREV_ROOT = Path.cwd().resolve()
ALT_ABBREV_OUTPUT_PATH = ALT_ABBREV_ROOT / "output"

In [12]:
path = Path(
    ALT_ABBREV_OUTPUT_PATH
    / "llm_runs"
    / "stored_runs_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3.pkl"
)

with open(path, "rb") as f:
    data = pickle.load(f)

all_runs = data["runs"]

## Summary of runs

In [13]:
sample_names = set()

unique_runs = {
    (
        run["temperature"],
        run["run_idx"],
    )
    for run in all_runs
}
print(f"Total number of runs stored: {len(all_runs)}")
print(f"Unique runs: {unique_runs}")

Total number of runs stored: 3
Unique runs: {(0.0, 1), (0.0, 2), (0.0, 0)}


## Format runs for analysis

In [14]:
if isinstance(all_runs, dict):
    all_runs = all_runs["runs"]

rows = []

for run in all_runs:
    for i, r in enumerate(run["results"]):
        rows.append(
            {
                "row": i,
                "run": run["run_idx"],
                "temp": run["temperature"],
                "llm": r["llm_annotation"],
                "gt": r["gt"],
            }
        )

df_runs = pl.DataFrame(rows).with_columns(
    [
        pl.col("llm").cast(pl.Boolean),
        pl.col("gt").cast(pl.Boolean),
    ]
)

In [15]:
results_by_run = {}

for temp, run_idx in unique_runs:
    run_df = df_runs.filter((pl.col("temp") == temp) & (pl.col("run") == run_idx))

    (
        llm_summary,
        llm_acc,
        llm_metrics,
        llm_cov,
        system_summary,
        system_acc,
        system_metrics,
    ) = analyze_results(run_df)

    run_result = RunResult(
        temperature=temp,
        run_idx=run_idx,
        llm_accuracy=llm_acc,
        llm_coverage=llm_cov,
        llm_summary=llm_summary,
        llm_metrics=llm_metrics,
        system_accuracy=system_acc,
        system_summary=system_summary,
        system_metrics=system_metrics,
    )

    results_by_run[(temp, run_idx)] = run_result

## Analysis between runs

In [16]:
all_results = []

for run in results_by_run.values():
    all_results.append(
        {
            "temperature": run.temperature,
            "run_idx": run.run_idx,
            "llm_accuracy": run.llm_accuracy,
            "llm_coverage": run.llm_coverage,
            "system_accuracy": run.system_accuracy,
        }
    )

results_df = pl.DataFrame(all_results)
print(results_df)

shape: (3, 5)
┌─────────────┬─────────┬──────────────┬──────────────┬─────────────────┐
│ temperature ┆ run_idx ┆ llm_accuracy ┆ llm_coverage ┆ system_accuracy │
│ ---         ┆ ---     ┆ ---          ┆ ---          ┆ ---             │
│ f64         ┆ i64     ┆ f64          ┆ f64          ┆ f64             │
╞═════════════╪═════════╪══════════════╪══════════════╪═════════════════╡
│ 0.0         ┆ 1       ┆ 0.886364     ┆ 0.435644     ┆ 0.950495        │
│ 0.0         ┆ 2       ┆ 0.895455     ┆ 0.435644     ┆ 0.954455        │
│ 0.0         ┆ 0       ┆ 0.895455     ┆ 0.435644     ┆ 0.954455        │
└─────────────┴─────────┴──────────────┴──────────────┴─────────────────┘


## Analysis of a single run

In [17]:
run_00_2 = results_by_run[(0.0, 2)]
print(f"LLM Coverage: {run_00_2.llm_coverage}")
print(run_00_2.llm_metrics)
print(run_00_2.llm_accuracy)
print(run_00_2.system_metrics)
print(run_00_2.system_accuracy)

LLM Coverage: 0.43564356435643564
shape: (1, 8)
┌─────────────────┬───────────┬──────────┬──────────┬─────┬─────┬─────┬─────┐
│ evaluation_type ┆ precision ┆ recall   ┆ f1       ┆ TP  ┆ FP  ┆ TN  ┆ FN  │
│ ---             ┆ ---       ┆ ---      ┆ ---      ┆ --- ┆ --- ┆ --- ┆ --- │
│ str             ┆ f64       ┆ f64      ┆ f64      ┆ i64 ┆ i64 ┆ i64 ┆ i64 │
╞═════════════════╪═══════════╪══════════╪══════════╪═════╪═════╪═════╪═════╡
│ conditional_llm ┆ 0.789474  ┆ 0.895522 ┆ 0.839161 ┆ 60  ┆ 16  ┆ 137 ┆ 7   │
└─────────────────┴───────────┴──────────┴──────────┴─────┴─────┴─────┴─────┘
0.8954545454545455
shape: (1, 8)
┌─────────────────┬───────────┬──────────┬──────────┬─────┬─────┬─────┬─────┐
│ evaluation_type ┆ precision ┆ recall   ┆ f1       ┆ TP  ┆ FP  ┆ TN  ┆ FN  │
│ ---             ┆ ---       ┆ ---      ┆ ---      ┆ --- ┆ --- ┆ --- ┆ --- │
│ str             ┆ f64       ┆ f64      ┆ f64      ┆ i64 ┆ i64 ┆ i64 ┆ i64 │
╞═════════════════╪═══════════╪══════════╪══════════╪═════╪══

In [18]:
df = pl.read_parquet(path.with_suffix(".parquet"))

## Evaluation df for troubleshooting and idea generation

In [19]:
eval_df = build_eval_df(
    df,
    all_runs,
    temperature=0.0,
    run_idx=1,
)
eval_df

HGNC_ID,ENSG_ID,NCBI_ID,captured_status,captured_category_list,primary_gene_symbol,gene_symbol,alternate_abbreviation_status,gene_name,llm,matched_rule,temperature,run_idx,llm_correct
str,str,str,bool,str,str,str,bool,str,bool,str,f64,i64,bool
"""HGNC:12406""","""ENSG00000166402""","""GENE ID:7275""",false,null,"""TUB""","""rd5""",false,"""TUB bipartite transcription fa…",null,"""low_lcs_similarity""",0.0,1,null
"""HGNC:6631""","""ENSG00000074695""","""GENE ID:3998""",false,null,"""LMAN1""","""GP58""",false,"""lectin, mannose binding 1""",null,"""extra_characters""",0.0,1,null
"""HGNC:9911""","""ENSG00000216835""","""GENE ID:3186""",false,null,"""RBMXP1""","""HNRNP-G""",false,"""RBMX pseudogene 1""",false,null,0.0,1,true
"""HGNC:3795""","""ENSG00000110203""","""GENE ID:2352""",false,null,"""FOLR3""","""FRgamma""",true,"""folate receptor gamma""",true,null,0.0,1,true
"""HGNC:13919""","""ENSG00000234651, ENSG000000961…","""GENE ID:7917""",false,null,"""BAG6""","""SCYTHE""",false,"""BAG cochaperone 6""",null,"""extra_characters""",0.0,1,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HGNC:29334""","""ENSG00000159733""","""GENE ID:57732""",false,null,"""ZFYVE28""","""LYST2""",false,"""zinc finger FYVE-type containi…",false,null,0.0,1,true
"""HGNC:43475""","""ENSG00000284092""","""GENE ID:100847081""",true,"""Prefix Gene Group Symbol""","""MIR5703""","""mir-5703""",true,"""microRNA 5703""",true,null,0.0,1,true
"""HGNC:22423""","""ENSG00000146858""","""GENE ID:92092""",true,"""Previous Symbol, Placeholder S…","""ZC3HAV1L""","""C7orf39""",false,"""ZC3HAV1 like""",null,"""extra_characters""",0.0,1,null
